In [30]:
pip install huggingface datasets pandas scikit-learn pyidaungsu



[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [31]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd
import re
import numpy as np

dataset = load_dataset("kalixlouiis/myanmar-sentiment-analysis")
df = pd.DataFrame(dataset['train'])

print(f"Dataset shape: {df.shape}")
print(f"Class distribution:\n{df['label'].value_counts()}")

Dataset shape: (1665, 2)
Class distribution:
label
Positive    555
Negative    555
Neutral     555
Name: count, dtype: int64


In [32]:

def segment_burmese_syllables(text):
    """Simple rule-based Burmese syllable segmenter."""
    consonant_pattern = r'[က-အ]'
    segments = []
    current = ""
    
    for char in text:
        if re.match(consonant_pattern, char) and not current.endswith('္'):
            if current:
                segments.append(current)
            current = char
        else:
            current += char
    
    if current:
        segments.append(current)
    
    return " ".join(segments)

df['segmented_text'] = df['text'].apply(segment_burmese_syllables)


In [33]:

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95
)

X = vectorizer.fit_transform(df['segmented_text'])

# IMPORTANT: Keep labels as integers
y = df['label']  # These should be 0, 1, 2


In [34]:
# ==================== STEP 4: TRAIN-TEST SPLIT ====================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training samples: {X_train.shape[0]}")
print(f"Test samples: {X_test.shape[0]}")

Training samples: 1332
Test samples: 333


In [35]:

# Model 1: Linear SVM
svc = SVC(kernel='linear', class_weight='balanced', random_state=42)
svc.fit(X_train, y_train)
y_pred_svc = svc.predict(X_test)

# Model 2: Logistic Regression
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Model 3: Random Forest
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

In [36]:
print("=" * 60)
print("MODEL PERFORMANCE COMPARISON")
print("=" * 60)

acc_svc = accuracy_score(y_test, y_pred_svc)
acc_lr = accuracy_score(y_test, y_pred_lr)
acc_rf = accuracy_score(y_test, y_pred_rf)

print(f"SVM Accuracy:              {acc_svc:.4f}")
print(f"Logistic Regression Acc:   {acc_lr:.4f}")
print(f"Random Forest Acc:         {acc_rf:.4f}")


MODEL PERFORMANCE COMPARISON
SVM Accuracy:              0.3483
Logistic Regression Acc:   0.3544
Random Forest Acc:         0.3544


In [37]:
models = {
    'SVM': (svc, y_pred_svc),
    'Logistic Regression': (lr, y_pred_lr),
    'Random Forest': (rf, y_pred_rf)
}

for name, (model, y_pred) in models.items():
    print("\n" + "=" * 60)
    print(f"{name} - Classification Report")
    print("=" * 60)
    print(classification_report(y_test, y_pred, 
                               target_names=['Negative', 'Positive', 'Neutral']))



SVM - Classification Report
              precision    recall  f1-score   support

    Negative       0.50      0.01      0.02       111
    Positive       0.62      0.05      0.08       111
     Neutral       0.34      0.99      0.51       111

    accuracy                           0.35       333
   macro avg       0.49      0.35      0.20       333
weighted avg       0.49      0.35      0.20       333


Logistic Regression - Classification Report
              precision    recall  f1-score   support

    Negative       0.50      0.01      0.02       111
    Positive       0.70      0.06      0.12       111
     Neutral       0.34      0.99      0.51       111

    accuracy                           0.35       333
   macro avg       0.51      0.35      0.21       333
weighted avg       0.51      0.35      0.21       333


Random Forest - Classification Report
              precision    recall  f1-score   support

    Negative       0.50      0.01      0.02       111
    Positive    

In [38]:

def predict_sentiment(text, model, vectorizer):
    """Predict sentiment for a new Myanmar sentence."""
    # Segment the text
    segmented = segment_burmese_syllables(text)
    
    # Transform using the trained vectorizer
    features = vectorizer.transform([segmented])
    
    # Predict
    pred = model.predict(features)[0]
    
    # Handle both integer and string predictions
    if isinstance(pred, (int, np.integer)):
        label_map = {0: "Negative", 1: "Positive", 2: "Neutral"}
        return label_map[pred]
    else:
        # If it's already a string, return it directly
        return str(pred)



In [ ]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# 1. Custom Syllable Segmenter
def segment_burmese_syllables(text):
    if not isinstance(text, str):
        return ""
    consonant_pattern = r'[က-အ]'
    segments = []
    current = ""
    for char in text:
        if re.match(consonant_pattern, char) and current and not current.endswith(('္', '်')):
            segments.append(current)
            current = char
        else:
            current += char
    if current:
        segments.append(current)
    return " ".join(segments)

# Custom tokenizer adapter for TfidfVectorizer
def custom_tokenizer(text):
    segmented = segment_burmese_syllables(text)
    return segmented.split()

# 2. Build Pipeline (Tokenizer integrated directly)
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        tokenizer=custom_tokenizer,
        token_pattern=None,  # Suppress default regex warning
        max_features=5000,
        min_df=1,            # Set min_df=1 if dataset is small
        ngram_range=(1, 2)
    )),
    ('clf', LogisticRegression(class_weight='balanced', random_state=42))
])

# 3. Train Pipeline directly on RAW text strings (Do NOT segment beforehand)
pipeline.fit(df['text'], df['label'])

# 4. Corrected Prediction Function
def predict_sentiment(text, model_pipeline):
    # Pass raw unsegmented text — Pipeline handles tokenization automatically
    pred = model_pipeline.predict([text])[0]
    probabilities = model_pipeline.predict_proba([text])[0]
    confidence = float(np.max(probabilities))
    
    label_map = {0: "Negative", 1: "Positive", 2: "Neutral"}
    return {
        "sentiment": label_map.get(pred, str(pred)),
        "confidence": round(confidence, 4)
    }



Text:      ဒီဇာတ်လမ်းက တကယ်ကောင်းတယ်။
True:      Positive
Predicted: Positive (Confidence: 0.7573)
--------------------------------------------------
Text:      ဝန်ဆောင်မှုက အရမ်းဆိုးတယ်။
True:      Negative
Predicted: Negative (Confidence: 0.6892)
--------------------------------------------------
Text:      ရုံးပိတ်ရက်ဖြစ်လို့ ရုံးခန်းတွေ အားလုံးပိတ်ထားပါတယ်။
True:      Neutral
Predicted: Neutral (Confidence: 0.7876)
--------------------------------------------------


In [51]:
# 5. Run Test
test_sentences = [
    # Positive
    {"text": "ဒီအထည်က သားနားပြီး အသားလည်း အရမ်းအေးတယ်။", "true_label": 1, "true_name": "Positive"},
    {"text": "ပစ္စည်း လျင်မြန်စွာ ရောက်ရှိလာလို့ အရမ်းကျေနပ်မိပါတယ်။", "true_label": 1, "true_name": "Positive"},
    {"text": "ဝန်ထမ်းတွေရဲ့ ပြောဆိုဆက်ဆံပုံက အရမ်းယဉ်ကျေးပါတယ်။", "true_label": 1, "true_name": "Positive"},
    
    # Negative
    {"text": "မှာထားတာနဲ့ တခြားစီ ရောက်လာတယ် စိတ်ပျက်စရာပဲ။", "true_label": 0, "true_name": "Negative"},
    {"text": "ဈေးနှုန်းက ခေါင်ခိုက်နေပေမယ့် အရည်အသွေးက လုံးဝမကောင်းဘူး။", "true_label": 0, "true_name": "Negative"},
    {"text": "အဆင်မပြေဘူး နောက်တစ်ခါ ဘယ်တော့မှ မဝယ်တော့ပါဘူး။", "true_label": 0, "true_name": "Negative"},
    {"text": "အစားအသောက်က သိုးနေပြီး လတ်ဆတ်မှု လုံးဝမရှိဘူး။", "true_label": 0, "true_name": "Negative"},
    
    # Neutral
    {"text": "ဆိုင်ကို မနက် ၉ နာရီကနေ ည ၈ နာရီအထိ ဖွင့်လှစ်ထားပါတယ်။", "true_label": 2, "true_name": "Neutral"},
    {"text": "ဒီနေ့ ရန်ကုန်မြို့မှာ မိုးရွာသွန်းနိုင်ခြေ ရှိပါတယ်။", "true_label": 2, "true_name": "Neutral"},
    {"text": "ပစ္စည်း ပို့ဆောင်ခ ဘယ်လောက်ကျသင့်လဲ သိချင်ပါတယ်။", "true_label": 2, "true_name": "Neutral"}
]

for sample in test_sentences:
    result = predict_sentiment(sample["text"], pipeline)
    print(f"Text:      {sample['text']}")
    print(f"True:      {sample['true_name']}")
    print(f"Predicted: {result['sentiment']} (Confidence: {result['confidence']})")
    print("-" * 50)

Text:      ဒီအထည်က သားနားပြီး အသားလည်း အရမ်းအေးတယ်။
True:      Positive
Predicted: Positive (Confidence: 0.5334)
--------------------------------------------------
Text:      ပစ္စည်း လျင်မြန်စွာ ရောက်ရှိလာလို့ အရမ်းကျေနပ်မိပါတယ်။
True:      Positive
Predicted: Positive (Confidence: 0.6472)
--------------------------------------------------
Text:      ဝန်ထမ်းတွေရဲ့ ပြောဆိုဆက်ဆံပုံက အရမ်းယဉ်ကျေးပါတယ်။
True:      Positive
Predicted: Negative (Confidence: 0.6934)
--------------------------------------------------
Text:      မှာထားတာနဲ့ တခြားစီ ရောက်လာတယ် စိတ်ပျက်စရာပဲ။
True:      Negative
Predicted: Negative (Confidence: 0.68)
--------------------------------------------------
Text:      ဈေးနှုန်းက ခေါင်ခိုက်နေပေမယ့် အရည်အသွေးက လုံးဝမကောင်းဘူး။
True:      Negative
Predicted: Negative (Confidence: 0.8776)
--------------------------------------------------
Text:      အဆင်မပြေဘူး နောက်တစ်ခါ ဘယ်တော့မှ မဝယ်တော့ပါဘူး။
True:      Negative
Predicted: Negative (Confidence: 0.823)
------------------

Text:       ဒီဇာတ်လမ်းက တကယ်ကောင်းတယ်။
True:       Positive
Predicted:  Positive (Confidence: 0.3368)
--------------------------------------------------
Text:       ဝန်ဆောင်မှုက အရမ်းဆိုးတယ်။
True:       Negative
Predicted:  Positive (Confidence: 0.3368)
--------------------------------------------------
Text:       ရုံးပိတ်ရက်ဖြစ်လို့ ရုံးခန်းတွေ အားလုံးပိတ်ထားပါတယ်။
True:       Neutral
Predicted:  Positive (Confidence: 0.3368)
--------------------------------------------------
